In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

%cd /content/drive/MyDrive/


In [ ]:
%cd /content/drive/MyDrive/

In [3]:
# If running on Colab, install minimal deps
# (Comment out torch install if Colab runtime already has a good CUDA build.)
# %pip install -q --upgrade transformers accelerate

import os, sys, platform, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.8.0+cu126
CUDA available: True


In [4]:
!pip install llamafactory

In [5]:
#!pip -q install --upgrade "transformers>=4.43" accelerate safetensors sentencepiece einops
import torch, platform, sys, os, textwrap
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


Torch: 2.8.0+cu126 | CUDA: True | Device: NVIDIA A100-SXM4-80GB


In [6]:
# Versions + sanity checks (run after installing deps)

import sys, os, platform, tempfile, warnings
import torch
from packaging import version

# Core libs
import transformers, accelerate, safetensors, sentencepiece, einops

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
print("Torch:", torch.__version__, "| CUDA avail:", torch.cuda.is_available(), "| CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU CC:", f"{props.major}.{props.minor}", "| Total VRAM (GB):", round(props.total_memory/1024**3, 2))

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("safetensors:", safetensors.__version__)
print("sentencepiece:", sentencepiece.__version__)
print("einops:", einops.__version__)

# Assert min versions you requested
assert version.parse(transformers.__version__) >= version.parse("4.43"), "Transformers >= 4.43 required"


Python: 3.12.12 | Platform: Linux-6.6.105+-x86_64-with-glibc2.35
Torch: 2.8.0+cu126 | CUDA avail: True | CUDA: 12.6
GPU: NVIDIA A100-SXM4-80GB
GPU CC: 8.0 | Total VRAM (GB): 79.32
Transformers: 4.52.4
Accelerate: 1.7.0
safetensors: 0.6.2
sentencepiece: 0.2.1
einops: 0.8.1


In [ ]:
BASE_MODEL_NAME        = "Qwen/Qwen2.5-0.5B-Instruct"     # LLM backbone
PROTEIN_CONFIG = "/weights/ProTrek_35M/esm2_t12_35M_UR50D"
STRUCTURE_CONFIG = "/weights/ProTrek_35M/foldseek_t12_35M"
PROTREK_CKPT_PATH    = "/weights/ProTrek_35M/ProTrek_35M.pt"


DTYPE = "bf16" if torch.cuda.is_available() else "fp16"
print("Using dtype:", DTYPE)


Using dtype: bf16


In [8]:

# Import your updated PLLM module (must be in the same folder)
#import protein_llm  # this file must define class `PLLM`
import protein_llm.models.modeling_pllm

[WARN] flash_attn is not installed, flash_attn will not work


In [9]:
import os, json, math
from typing import Optional, List
import torch
from transformers import AutoTokenizer

# Import your updated PLLM module (must be in the same folder)
import protein_llm  # this file must define class `PLLM`
import protein_llm.models.modeling_pllm as PLLM
from protein_llm.models.configuration_pllm import PLLMConfig

# ===== USER-EDITABLE PATHS =====
#BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # or "Qwen/Qwen2.5-0.5B"
PROT_SLOT = 1
STRU_SLOT = 3

# BF16 is optimal for A100
DTYPE_STR = "bf16"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
tokenizer.add_tokens("<protein>")
tokenizer.add_tokens("<structure>")
protein_token_id = tokenizer("<protein>", add_special_tokens=False).input_ids[-1]
structure_token_id = tokenizer("<structure>", add_special_tokens=False).input_ids[-1]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [11]:
cfg = PLLMConfig(
    base_model_name_or_path=BASE_MODEL_NAME,
    protein_config=PROTEIN_CONFIG,
    structure_config=STRUCTURE_CONFIG,
    protrek_ckpt=PROTREK_CKPT_PATH if PROTREK_CKPT_PATH else None,
    prot_slot=1, stru_slot=3,
    single_token_prefix=False, prefix_len=4,
    proj_hid=1024, dropout=0.10,
    train_encoders=False,        # freeze encoders for the test
    load_pretrained=True,        # load base LLM from hub
    protein_token_id=protein_token_id,
    structure_token_id=structure_token_id,
)


In [12]:
pllm = PLLM.PLLM(cfg)
pllm.load_protrek_weights()
pllm = pllm.to("cuda")

[PLLM] Loaded generation_config from base model: Qwen/Qwen2.5-0.5B-Instruct
[ProteinEncoder] loaded from slot 1 | missing=[] unexpected=[]
[StructureEncoder] loaded from slot 3 | missing=[] unexpected=[]


In [13]:
user_prompt = (
    "You are a professional protein biologist. "
    "Based only on the provided inputs, produce a natural, concise, and biologically accurate description of the protein. "
    "First reason step by step inside a <thinking> block using sequence-derived evidence and structural cues, "
    "then provide the final 2–4 sentence description inside an <answer> block.\n\n"
    "Inputs:\n"
    "Protein: <protein>\n"
    "Structure: <structure>"
)

aa_seq = "MSKGTPSRGKRQTQTHLTCRRCGRMSYHKRHKICSSCGFGRSTRMRSYGWITKRPKVATH"
structure = "<|chain:A|> <|chain_sep|> #ddddvvvvpppddqfdqdppprdraqgpvqragpqqggpndpggdddpvvddddpdddd"


In [14]:
enc = tokenizer(user_prompt, return_tensors="pt")
input_ids = enc["input_ids"].to(DEVICE)
attention_mask = enc["attention_mask"].to(DEVICE)

print("Prompt length:", input_ids.shape[-1])
print("Has <protein>:", (input_ids == cfg.protein_token_id).any().item())
print("Has <structure>:", (input_ids == cfg.structure_token_id).any().item())

Prompt length: 77
Has <protein>: True
Has <structure>: True


In [15]:
chat_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": user_prompt}],
    add_generation_prompt=True,
    tokenize=False,
)

# Make sure the placeholders are present after templating
assert "<protein>" in chat_prompt and "<structure>" in chat_prompt, "Placeholders not found in chat prompt."

test_inputs = tokenizer(chat_prompt, return_tensors="pt").to(DEVICE)

# --- Generate (condition on aa_seq & structure) ---
pllm.eval()
with torch.no_grad():
    gen_ids = pllm.generate(
        **test_inputs,
        aa_seq=[aa_seq],
        stru_str=[structure],
        max_new_tokens=512,
        do_sample=False,      # deterministic for debugging; change if you want sampling
        temperature=1.0,
        top_p=1.0,
    )

# --- Strip the prompt tokens from the output and decode ---
trimmed = [
    output_ids[len(inp_ids):]
    for inp_ids, output_ids in zip(test_inputs.input_ids, gen_ids)
]
response = tokenizer.batch_decode(trimmed, skip_special_tokens=True)[0]

print("=== MODEL RESPONSE ===")
print(response)
print("\n(first 64 generated token ids):")
print(trimmed[0][:64].tolist())

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== MODEL RESPONSE ===
<thinking>
The provided input describes a protein structure with multiple subunits arranged in a complex network. The first part indicates that there are five main subunits (each represented by "5" in the sequence), which suggests this is likely a multi-subunit protein. The second part mentions that each subunit has specific functional domains or regions, indicating a hierarchical organization within the protein's structure.
</thinking>

<answer>
This protein appears to be composed of five main subunits, each with distinct functional domains. The detailed information about each subunit's composition and function provides insight into its overall structure and potential biological roles.

(first 64 generated token ids):
[27, 82260, 397, 785, 3897, 1946, 16555, 264, 12833, 5944, 448, 5248, 1186, 25643, 27802, 304, 264, 6351, 3922, 13, 576, 1156, 949, 14807, 429, 1052, 525, 4236, 1887, 1186, 25643, 320, 9547, 15251, 553, 330, 20, 1, 304, 279, 8500, 701, 892, 13230, 

In [16]:
pllm.eval()
with torch.no_grad():
    out = pllm(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=None,                  # not needed for this test
        aa_seq=[aa_seq],
        stru_str=[structure],
        extract_qkv=True,             # <-- caches inputs_embeds & attention_mask for export
    )

print("Forward OK. Cached tensors ready for export.")


Forward OK. Cached tensors ready for export.


In [26]:
out

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 3.8906,  5.1250,  2.2969,  ..., -2.0625, -2.0625, -2.0625],
         [ 5.8750, -0.6562, -2.2188,  ..., -4.5312, -4.5312, -4.5312],
         [ 6.8125, -0.6289, -0.7617,  ..., -3.9062, -3.9062, -3.9062],
         ...,
         [ 4.8125,  6.1250,  2.6406,  ..., -3.4375, -3.4375, -3.4375],
         [ 4.7500,  6.4062,  3.2500,  ..., -3.9531, -3.9531, -3.9531],
         [ 4.5625,  5.8125,  3.2344,  ..., -3.1562, -3.1562, -3.1562]]],
       device='cuda:0', dtype=torch.bfloat16), past_key_values=None, hidden_states=None, attentions=None)

In [18]:
# Try layer 0; you can change to another valid index for your backbone
single = pllm.export_qkv(layer_idx=0, split_heads=False, detach=True, to_cpu=True)

q, k, v, m = single["q"], single["k"], single["v"], single["m"]
print("Single-layer export (no head split):")
print("  q shape:", tuple(q.shape))
print("  k shape:", tuple(k.shape))
print("  v shape:", tuple(v.shape))
print("  mask shape:", None if m is None else tuple(m.shape))
print("  meta:", single["meta"])


Single-layer export (no head split):
  q shape: (1, 213, 896)
  k shape: (1, 213, 128)
  v shape: (1, 213, 128)
  mask shape: (1, 213)
  meta: {'layer_idx': 0, 'split_heads': False, 'num_heads': 14, 'head_dim': 64, 'seq_len': 213}


In [21]:
# Try layer 0; you can change to another valid index for your backbone
single = pllm.export_qkv(layer_idx=[0,1,2,3], split_heads=False, detach=True, to_cpu=True)

q, k, v, m = single["q"], single["k"], single["v"], single["m"]
print("Single-layer export (no head split):")
print("  q shape:", tuple(q.shape))
print("  k shape:", tuple(k.shape))
print("  v shape:", tuple(v.shape))
print("  mask shape:", None if m is None else tuple(m.shape))
print("  meta:", single["meta"])

Single-layer export (no head split):
  q shape: (1, 213, 896)
  k shape: (1, 213, 128)
  v shape: (1, 213, 128)
  mask shape: (1, 213)
  meta: {'layer_idx': 3, 'split_heads': False, 'num_heads': 14, 'head_dim': 64, 'seq_len': 213}


In [27]:
# ---- Multi-layer export (no head split) ----
layers = [0, 1, 2, 3]
multi = pllm.export_qkv(layer_idx=layers, split_heads=False, detach=True, to_cpu=True)

print("Shared mask shape:", None if multi["m"] is None else tuple(multi["m"].shape))
for li in layers:
    blob = multi["layers"][li]
    q, k, v = blob["q"], blob["k"], blob["v"]
    print(f"[Layer {li}] q {tuple(q.shape)} | k {tuple(k.shape)} | v {tuple(v.shape)}")
    print("  meta:", blob["meta"])

Shared mask shape: (1, 213)
[Layer 0] q (1, 213, 896) | k (1, 213, 128) | v (1, 213, 128)
  meta: {'layer_idx': 0, 'split_heads': False, 'num_heads': 14, 'head_dim': 64, 'seq_len': 213}
[Layer 1] q (1, 213, 896) | k (1, 213, 128) | v (1, 213, 128)
  meta: {'layer_idx': 1, 'split_heads': False, 'num_heads': 14, 'head_dim': 64, 'seq_len': 213}
[Layer 2] q (1, 213, 896) | k (1, 213, 128) | v (1, 213, 128)
  meta: {'layer_idx': 2, 'split_heads': False, 'num_heads': 14, 'head_dim': 64, 'seq_len': 213}
[Layer 3] q (1, 213, 896) | k (1, 213, 128) | v (1, 213, 128)
  meta: {'layer_idx': 3, 'split_heads': False, 'num_heads': 14, 'head_dim': 64, 'seq_len': 213}


In [29]:
cfg = pllm.llm.config
layers = pllm.llm.model.layers  # Qwen2.* keeps decoder blocks here

print("Model type:", type(pllm.llm).__name__)
print("Hidden size:", cfg.hidden_size)
print("num_hidden_layers (from config):", getattr(cfg, "num_hidden_layers", "n/a"))
print("num_attention_heads:", getattr(cfg, "num_attention_heads", "n/a"))
print("num_key_value_heads:", getattr(cfg, "num_key_value_heads", getattr(cfg, "num_attention_heads", "n/a")))

n_layers = len(layers)
print("Actual module layers:", n_layers)
print("Valid layer indices:", list(range(n_layers)), "...")  # print first few

Model type: Qwen2ForCausalLM
Hidden size: 896
num_hidden_layers (from config): 24
num_attention_heads: 14
num_key_value_heads: 2
Actual module layers: 24
Valid layer indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23] ...


In [30]:
n_layers = len(pllm.llm.model.layers)
print(f"Total layers: {n_layers}")

# Show first 2 and last 2 layer types + that they have self_attn
for i in [0, 1, n_layers-2, n_layers-1]:
    blk = pllm.llm.model.layers[i]
    print(f"[{i}] block={type(blk).__name__}, has self_attn? {hasattr(blk, 'self_attn')}")

Total layers: 24
[0] block=Qwen2DecoderLayer, has self_attn? True
[1] block=Qwen2DecoderLayer, has self_attn? True
[22] block=Qwen2DecoderLayer, has self_attn? True
[23] block=Qwen2DecoderLayer, has self_attn? True
